## Introduction
This notebook is a submission to **Grab AI For Sea Challenge - Traffic Management**, to forecast travel demand based on historical Grab bookings. 
Challenge website: https://www.aiforsea.com/traffic-management

There are **four parts** in this notebook:
* **Data cleaning & preprocessing**
* **Model selection: Random Forest vs. XGBoost**
* **Define a function to predict demands of T+1, ..., T+5 using known data till T**
* **Predict demands of T+1, ..., T+5 using test data.** 

The test dataset can start from any time period after the timeframe of the training dataset. My model will use features from the test dataset ending at timestamp T and predict T+1 to T+5 for all the geohashes which appeared in the training dataset. 

Each time interval in this challenge is 15 minutes.

**For evaluators**: please uncomment the code in Part 4 and fill in the link of test dataset. The code will produce a CSV file containing the demand forecasts for T+1 to T+5 for all the geohashes from the training set. Please run all codes in this notebook to avoid any errors. 

In [1]:
import numpy as np
import pandas as pd
import os
print(os.listdir('./dataset'))


['sample_submission.csv', 'test.csv', 'train.csv']


## Part 1 - Data Cleaning & Preprocessing

Take a look at training set:

In [2]:
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

df_train = pd.read_csv('./dataset/train.csv')
df_train.head()


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


Size of training data:

In [3]:
df_train.shape

(77299, 11)

1329 unique locations in the data

In [4]:
len(df_train.geohash.unique())


1249

Convert timestamp into hours and mininutes:

In [5]:
df_train['hours'] = df_train['timestamp'].map(lambda x: int(x.split(':')[0]))
df_train['mins'] = df_train['timestamp'].map(lambda x: int(x.split(':')[1]))
df_train.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,hours,mins
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN,0,0
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,0,0
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,0,0
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy,0,0
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,0,0


Convert day, hours, mins into a single feature **"time"**:

In [6]:
df_train['time'] = 24*60*(df_train['day']-1) + 60*df_train['hours'] + df_train['mins']
df_train.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,hours,mins,time
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN,0,0,67680
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,0,0,67680
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,0,0,67680
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy,0,0,67680
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,0,0,67680


Convert geohash6 into latitude and longtitude:

In [7]:
import pygeohash as Geohash
df_train['Latitude'] = df_train.geohash.map(lambda x: float(Geohash.decode_exactly(x)[0]))
df_train['Longitude'] = df_train.geohash.map(lambda x: float(Geohash.decode_exactly(x)[1]))
df_train = df_train.sort_values(by=['time','Latitude','Longitude'], ascending=True)
df_train = df_train.reset_index().drop('index',axis=1)
df_train.head()


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,hours,mins,time,Latitude,Longitude
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN,0,0,67680,-5.484924,90.664673
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,0,0,67680,-5.462952,90.686646
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,0,0,67680,-5.462952,90.708618
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy,0,0,67680,-5.462952,90.862427
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,0,0,67680,-5.457458,90.675659


Not all locations appear in all time slots

In [8]:
df_train[['geohash','demand']].groupby('geohash').count().head(10)


,demand
geohash,
qp02yc,12
qp02yf,1
qp02yy,2
qp02yz,30
qp02z1,33
qp02z3,25
qp02z4,9
qp02z5,31
qp02z6,39


As the training set is a huge dataset with more than 4 million data, I will only use the last 14 days' data, out of which the last five timestamps are used for testing purpose and the rest is for training purpose.

In [9]:
max_day = df_train.day.max()
max_time = df_train.time.max()
train_start = df_train[df_train.day==48].index[0]
test_start = df_train[df_train.time >= max_time-15*4].index[0]

Xtrain = df_train[['time', 'Latitude','Longitude']].iloc[train_start:test_start,:]
Xtest = df_train[['time', 'Latitude','Longitude']].iloc[test_start:,:]

ytrain = df_train.demand.iloc[train_start:test_start]
ytest = df_train.demand.iloc[test_start:]


In [10]:
Xtrain.shape, Xtest.shape, ytrain.shape, ytest.shape

((72854, 3), (4445, 3), (72854,), (4445,))

## Part 2 - Model Selection

### Part 2.1 - RandomForestRegressor

In [11]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

model = RandomForestRegressor(n_estimators=30, max_depth=40)
model.fit(Xtrain, ytrain)
ytest_pred = model.predict(Xtest)
rmse = np.sqrt(mean_squared_error(ytest, ytest_pred))
print('RMSE:',rmse)

RMSE: 0.04739316496554


### Part 2.2 - XGBRegressor

In [12]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=35, n_jobs=-1)
model.fit(Xtrain, ytrain)
ytest_pred = model.predict(Xtest)
rmse = np.sqrt(mean_squared_error(ytest, ytest_pred))
print('RMSE:',rmse)


RMSE: 0.04751854940331488


#### From above output, XGBRegressor produces a smaller RMSE than RandomForestRegressor. Hence XGBRegressor will be used. 
#### All the hyperparameters above have been refined.[](http://)

Define a function to convert time into day, hour, minute and timestamp:

In [13]:
def convert_time(time):
    day = int(time/(24*60)) + 1
    hour = int((time-(day-1)*24*60)/60)
    minute = time-(day-1)*24*60-hour*60
    timestamp = ':'.join((str(hour),str(minute)))
    return (day, hour, minute, timestamp)

## Part 3 - Define a function to predict demands of T+1, ..., T+5 using known data till T 

In [14]:
def predict5ts(link, n_estimators=500, learning_rate=0.05, max_depth=35):
    df = pd.read_csv(link)
    df['hours'] = df['timestamp'].map(lambda x: int(x.split(':')[0]))
    df['mins'] = df['timestamp'].map(lambda x: int(x.split(':')[1]))
    df['time'] = 24*60*(df['day']-1) + 60*df['hours'] + df['mins']
    
    import pygeohash as Geohash
    df['Latitude'] = df.geohash.map(lambda x: float(Geohash.decode_exactly(x)[0]))
    df['Longitude'] = df.geohash.map(lambda x: float(Geohash.decode_exactly(x)[1]))

    df = df.sort_values(by=['time','Latitude','Longitude'], ascending=True)
    df = df.reset_index().drop('index',axis=1)
    
    X = df[['time', 'Latitude','Longitude']]
    y = df.demand
    
    from xgboost import XGBRegressor
    model = XGBRegressor(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, n_jobs=-1)
    model.fit(X, y)
    
    T = df.time.max()
    T1 = T+15
    T2 = T+15*2
    T3 = T+15*3
    T4 = T+15*4
    T5 = T+15*5
    
    geohashes = df_train.geohash.unique()
    geohashes2 = []
    latitudes = []
    longitudes = []
    times = []
    days = []
    timestamps = []

    for t in (T1,T2,T3,T4,T5):
        for gh in geohashes:
            geohashes2.append(gh)
            latitudes.append(float(Geohash.decode_exactly(gh)[0]))
            longitudes.append(float(Geohash.decode_exactly(gh)[1]))
            times.append(t)
            days.append(convert_time(t)[0])
            timestamps.append(convert_time(t)[-1])

    df_pred = pd.DataFrame({'geohash': geohashes2, 'day': days, 'timestamp': timestamps,
                        'time': times, 'Latitude': latitudes, 'Longitude': longitudes})
    Xtest = df_pred[['time', 'Latitude','Longitude']]
    ypred = model.predict(Xtest)

    df_pred['demand'] = ypred
    output = df_pred[['geohash', 'day', 'timestamp', 'demand']]
    output.to_csv('output.csv', index=False)


Check if the above function works by testing a small portion of data from the training set.

In [15]:
df_trial = df_train[['geohash','day','timestamp','demand']].iloc[-20000:,:]
df_trial.to_csv('df_trial.csv', index=False)

trial_link = 'df_trial.csv'
predict5ts(link=trial_link)

output = pd.read_csv('output.csv')
print(output.shape)
output.head()


(6245, 4)


,geohash,day,timestamp,demand
0,qp02z1,49,2:15,0.064395
1,qp02zt,49,2:15,0.034536
2,qp08bj,49,2:15,0.104287
3,qp08gt,49,2:15,0.003010
4,qp02zq,49,2:15,0.068902


In [16]:
os.remove("df_trial.csv")
os.remove("output.csv")

## Part 4 - Predict demands of T+1, ..., T+5 using test data
* Please uncomment below code and enter the link of test data.
* Below code will produce an output file **output.csv** which is the demand forecast of T+1,...,T+5 for all the geo-locations, where T is the last time stamp in the test data.

In [17]:
import pygeohash as Geohash
from xgboost import XGBRegressor

print('Training final XGBoost model on full training dataset (CPU)...')
Xtrain_all = df_train[['time', 'Latitude', 'Longitude']]
ytrain_all = df_train['demand']

final_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=35, n_jobs=-1)
final_model.fit(Xtrain_all, ytrain_all)

print('Preparing test data...')
df_test = pd.read_csv('./dataset/test.csv')
df_test['hours'] = df_test['timestamp'].map(lambda x: int(x.split(':')[0]))
df_test['mins'] = df_test['timestamp'].map(lambda x: int(x.split(':')[1]))
df_test['time'] = 24*60*(df_test['day']-1) + 60*df_test['hours'] + df_test['mins']
df_test['Latitude'] = df_test.geohash.map(lambda x: float(Geohash.decode_exactly(x)[0]))
df_test['Longitude'] = df_test.geohash.map(lambda x: float(Geohash.decode_exactly(x)[1]))

Xtest_final = df_test[['time', 'Latitude', 'Longitude']]

print('Predicting on test.csv...')
ypred_final = final_model.predict(Xtest_final)

df_test['demand'] = ypred_final
output = df_test[['Index', 'demand']]
output.to_csv('./predicted.csv', index=False)
print('Saved final predictions to predicted.csv!')
print(output.head(10))


Training final XGBoost model on full training dataset (CPU)...


Preparing test data...


Predicting on test.csv...


Saved final predictions to predicted.csv!
   Index    demand
0      0  0.064290
1      1  0.019021
2      2  0.053067
3      3  0.022592
4      4  0.051392
5      5  0.026256
6      6  0.039901
7      7  0.129047
8      8  0.057484
9      9  0.027971
